# spharmgrid — ERA5 850-hPa spherical harmonic explorer

Compare one ERA5 850-hPa wind frame before and after vector spherical-harmonic filtering or fixed-grid regridding. `Run All` starts at the first timestamp and remains on that frame until a control is changed. The notebook also works with `panel serve`.

In [ ]:
from functools import lru_cache
from pathlib import Path

import cartopy.crs as ccrs
import geoviews as gv
import holoviews as hv
import numpy as np
import panel as pn
import pooch
import xarray as xr

import spharmgrid as sg

pn.extension()
hv.extension("bokeh")

DATA_RELEASE = "v0.2.0-data"
DATA_BASE = (
    f"https://github.com/mwyau/PyStormTracker-Data/releases/download/{DATA_RELEASE}/"
)
UV_FILE = "era5_uv850_2025-2026_djf_2.5x2.5.nc"
VO_FILE = "era5_vo850_2025-2026_djf_2.5x2.5.nc"
REGISTRY = {
    UV_FILE: "sha256:43cbc346a52c5230ac34eb22c7a640800fbffad40da4058686c8042a76bc5965",
    VO_FILE: "sha256:46ce78cd3b065d3777c2d628cdc2311d68a9fcb4d3a3b9948db7c7376ae7a6aa",
}
DATA = pooch.create(
    path=Path(pooch.os_cache("spharmgrid")) / "interactive-v0.2.0-data",
    base_url=DATA_BASE,
    registry=REGISTRY,
)


def fetch(name: str) -> Path:
    return Path(DATA.fetch(name, progressbar=False))


def pressure_hpa(coord: xr.DataArray) -> float:
    value = float(coord.values[0])
    units = str(coord.attrs.get("units", "")).strip().lower()
    if units in {"pa", "pascal", "pascals"}:
        return value / 100.0
    if units in {
        "hpa",
        "hectopascal",
        "hectopascals",
        "mbar",
        "millibar",
        "millibars",
    }:
        return value
    if not units and np.isclose(value, 850.0):
        return value
    if not units and np.isclose(value, 85000.0):
        return value / 100.0
    raise ValueError(f"Unsupported pressure units {coord.attrs.get('units')!r}")


with xr.open_dataset(fetch(UV_FILE), engine="h5netcdf") as raw:
    if {"u", "v"} - set(raw.data_vars):
        raise ValueError("Pinned ERA5 file must contain u and v")
    if raw.sizes.get("pressure_level") != 1:
        raise ValueError("Expected one pressure level")
    if not np.isclose(pressure_hpa(raw["pressure_level"]), 850.0):
        raise ValueError("Expected the 850-hPa pressure level")
    ERA5 = raw[["u", "v"]].isel(pressure_level=0, drop=True).load()

TIMES = ERA5["valid_time"].values
source_grid = sg.detect_grid(ERA5["u"].isel(valid_time=0))
if source_grid.kind != "cc":
    raise ValueError(f"Expected CC grid, found {source_grid.kind!r}")


def triangular_limit(grid: sg.Grid) -> int:
    latitude_lmax = grid.nlat - 2 if grid.kind == "cc" else grid.nlat - 1
    return min(latitude_lmax, (grid.nlon - 1) // 2)


SOURCE_LMAX = triangular_limit(source_grid)
if SOURCE_LMAX < 42:
    raise ValueError(f"Pinned grid supports only T{SOURCE_LMAX}")

latitude_order = (
    "descending"
    if source_grid.latitude[0] > source_grid.latitude[-1]
    else "ascending"
)
gl_target = sg.gaussian_grid(
    source_grid.nlat - 1,
    source_grid.nlon,
    lon0=float(source_grid.longitude[0]),
    latitude_order=latitude_order,
)

{
    "grid": source_grid.kind.upper(),
    "shape": ERA5["u"].shape,
    "timestamps": len(TIMES),
    "first_timestamp": str(TIMES[0]),
    "spectral_limit": f"T{SOURCE_LMAX}",
    "gl_target": (gl_target.nlat, gl_target.nlon),
}

## Processing

`ERA5 CC` filtering calls `sg.regrid_vector` with the source grid as the target, so the geographic `u` and `v` components are transformed and filtered together as one vector field. `Gauss–Legendre` performs the same spectral operation while synthesizing to a 72×144 GL grid.

In [ ]:
DIAGNOSTICS = (
    "Wind",
    "Relative vorticity",
    "Divergence",
    "Streamfunction",
    "Velocity potential",
    "Rotational wind",
    "Divergent wind",
)
GRID_MODES = ("ERA5 CC", "Gauss–Legendre")


@lru_cache(maxsize=72)
def frame(index: int) -> tuple[xr.DataArray, xr.DataArray]:
    current = ERA5.isel(valid_time=int(index))
    return current["u"], current["v"]


@lru_cache(maxsize=36)
def processed_wind(
    index: int,
    lmin: int,
    lmax: int,
    taper: float | None,
    grid_name: str,
) -> xr.Dataset:
    target = source_grid if grid_name == "ERA5 CC" else gl_target
    max_degree = min(SOURCE_LMAX, triangular_limit(target))
    if not 0 <= lmin <= lmax <= max_degree:
        raise ValueError(f"Invalid spectral range T{lmin}-{lmax}")
    u, v = frame(index)
    return sg.regrid_vector(
        u,
        v,
        target,
        lmin=lmin,
        lmax=lmax,
        taper=taper,
    )


def diagnostic_state(u: xr.DataArray, v: xr.DataArray, diagnostic: str):
    if diagnostic == "Wind":
        return "wind", np.hypot(u, v), u, v, 1.0, "Wind speed (m s⁻¹)"

    if diagnostic in {"Relative vorticity", "Divergence"}:
        kin = sg.kinematics(u, v)
        if diagnostic == "Relative vorticity":
            return (
                "scalar",
                kin["vo"],
                None,
                None,
                1e5,
                "Relative vorticity (10⁻⁵ s⁻¹)",
            )
        return "scalar", kin["d"], None, None, 1e5, "Divergence (10⁻⁵ s⁻¹)"

    if diagnostic in {"Streamfunction", "Velocity potential"}:
        pot = sg.potentials(u, v)
        if diagnostic == "Streamfunction":
            return (
                "scalar",
                pot["strf"],
                None,
                None,
                1e-6,
                "Streamfunction (10⁶ m² s⁻¹)",
            )
        return (
            "scalar",
            pot["vp"],
            None,
            None,
            1e-6,
            "Velocity potential (10⁶ m² s⁻¹)",
        )

    kin = sg.kinematics(u, v)
    if diagnostic == "Rotational wind":
        wind = sg.rotational_wind(kin["vo"], quantity="vorticity")
        return (
            "wind",
            np.hypot(wind["u_rotational"], wind["v_rotational"]),
            wind["u_rotational"],
            wind["v_rotational"],
            1.0,
            "Rotational wind (m s⁻¹)",
        )

    wind = sg.divergent_wind(kin["d"], quantity="divergence")
    return (
        "wind",
        np.hypot(wind["u_divergent"], wind["v_divergent"]),
        wind["u_divergent"],
        wind["v_divergent"],
        1.0,
        "Divergent wind (m s⁻¹)",
    )


@lru_cache(maxsize=72)
def original_state(index: int, diagnostic: str):
    return diagnostic_state(*frame(index), diagnostic)


@lru_cache(maxsize=72)
def processed_state(
    index: int,
    lmin: int,
    lmax: int,
    taper: float | None,
    grid_name: str,
    diagnostic: str,
):
    wind = processed_wind(index, lmin, lmax, taper, grid_name)
    return diagnostic_state(wind["u"], wind["v"], diagnostic)

In [ ]:
MAP_CRS = ccrs.PlateCarree()
COASTLINE = gv.feature.coastline()
VECTOR_STRIDE = 4


def values(field: xr.DataArray) -> np.ndarray:
    return np.asarray(field.values, dtype=float)


def plotting_array(field: xr.DataArray):
    field = field.transpose("latitude", "longitude")
    longitude = (np.asarray(field["longitude"]) + 180.0) % 360.0 - 180.0
    order = np.argsort(longitude)
    return longitude[order], np.asarray(field["latitude"]), values(field)[:, order]


def plotting_wind(u: xr.DataArray, v: xr.DataArray):
    longitude, latitude, u_values = plotting_array(u)
    _, _, v_values = plotting_array(v)
    return longitude, latitude, u_values, v_values


def common_limits(original, processed):
    data = np.concatenate(
        (
            values(original[1]).ravel() * original[4],
            values(processed[1]).ravel() * processed[4],
        )
    )
    data = data[np.isfinite(data)]
    if original[0] == "scalar":
        limit = max(float(np.percentile(np.abs(data), 99.0)), 1e-12)
        return -limit, limit
    return 0.0, max(float(np.percentile(data, 99.0)), 1.0)


def map_options(title: str):
    return dict(
        width=620,
        height=360,
        xlim=(-180, 180),
        ylim=(-90, 90),
        projection=MAP_CRS,
        title=title,
        tools=["hover", "pan", "wheel_zoom", "reset"],
    )


def draw(state, clim, title: str):
    kind, field, u, v, scale, label = state
    longitude, latitude, data = plotting_array(field)
    mesh = gv.QuadMesh(
        (longitude, latitude, data * scale),
        kdims=["longitude", "latitude"],
        vdims=[label],
        crs=MAP_CRS,
    ).opts(
        cmap="RdBu_r" if kind == "scalar" else "Viridis",
        colorbar=True,
        clim=clim,
        **map_options(title),
    )
    coast = COASTLINE.opts(
        line_color="#374151",
        line_width=0.8,
        projection=MAP_CRS,
    )
    if kind == "scalar":
        return mesh * coast

    longitude, latitude, u_values, v_values = plotting_wind(u, v)
    display_longitude = longitude[::VECTOR_STRIDE]
    display_latitude = latitude[1:-1:VECTOR_STRIDE]
    display_u = u_values[1:-1:VECTOR_STRIDE, ::VECTOR_STRIDE]
    display_v = v_values[1:-1:VECTOR_STRIDE, ::VECTOR_STRIDE]
    vectors = gv.VectorField(
        (
            display_longitude,
            display_latitude,
            np.arctan2(display_v, display_u),
            np.hypot(display_u, display_v),
        ),
        kdims=["longitude", "latitude"],
        vdims=["angle", "magnitude"],
        crs=MAP_CRS,
    ).opts(
        color="#111827",
        line_width=1,
        pivot="mid",
        projection=MAP_CRS,
        xlim=(-180, 180),
        ylim=(-90, 90),
    )
    return mesh * vectors * coast


def render(
    side,
    index,
    spectral_range,
    taper_enabled,
    taper_value,
    grid_name,
    diagnostic,
):
    lmin, lmax = map(int, spectral_range)
    taper = float(taper_value) if taper_enabled else None
    original = original_state(int(index), diagnostic)
    processed = processed_state(
        int(index),
        lmin,
        lmax,
        taper,
        grid_name,
        diagnostic,
    )
    state = original if side == "original" else processed
    return draw(
        state,
        common_limits(original, processed),
        f"{side.title()} · {diagnostic}",
    )


def timestamp(index: int) -> str:
    return (
        np.datetime_as_string(TIMES[int(index)], unit="m").replace("T", " ")
        + " UTC"
    )

## Interactive application

`Run All` renders frame 0 and stays there. Time, spectral range, and taper-response sliders are throttled, so expensive transforms run only after a slider is released. There is no animation control in the default example.

In [ ]:
diagnostic_widget = pn.widgets.Select(
    name="Diagnostic",
    options=list(DIAGNOSTICS),
    value="Wind",
)
time_widget = pn.widgets.IntSlider(
    name="Time index",
    start=0,
    end=len(TIMES) - 1,
    value=0,
    step=1,
)
spectral_widget = pn.widgets.IntRangeSlider(
    name="Spectral degree range",
    start=0,
    end=SOURCE_LMAX,
    value=(0, 42),
    step=1,
)
taper_enabled = pn.widgets.Checkbox(name="Taper enabled", value=False)
taper_value = pn.widgets.FloatSlider(
    name="Taper endpoint response",
    start=0.01,
    end=1.0,
    step=0.01,
    value=0.1,
    disabled=True,
)
grid_widget = pn.widgets.Select(
    name="Output grid",
    options=list(GRID_MODES),
    value="ERA5 CC",
)


def toggle_taper(event):
    taper_value.disabled = not bool(event.new)


taper_enabled.param.watch(toggle_taper, "value")

bindings = dict(
    index=time_widget.param.value_throttled,
    spectral_range=spectral_widget.param.value_throttled,
    taper_enabled=taper_enabled.param.value,
    taper_value=taper_value.param.value_throttled,
    grid_name=grid_widget.param.value,
    diagnostic=diagnostic_widget.param.value,
)
left_plot = hv.DynamicMap(
    pn.bind(render, side="original", **bindings),
    kdims=[],
)
right_plot = hv.DynamicMap(
    pn.bind(render, side="processed", **bindings),
    kdims=[],
)


def summary(index, spectral_range, enabled, endpoint, grid_name):
    lmin, lmax = map(int, spectral_range)
    filtering = (
        f"Sardeshmukh–Hoskins endpoint = {float(endpoint):g}"
        if enabled
        else "hard selection"
    )
    return (
        f"**{timestamp(index)}** · `T{lmin}–{lmax}` · "
        f"{filtering} · `{grid_name}`"
    )


timestamp_pane = pn.pane.Markdown(
    pn.bind(
        lambda index: f"**ERA5 timestamp:** `{timestamp(index)}`",
        time_widget.param.value_throttled,
    )
)
summary_pane = pn.pane.Markdown(
    pn.bind(
        summary,
        time_widget.param.value_throttled,
        spectral_widget.param.value_throttled,
        taper_enabled.param.value,
        taper_value.param.value_throttled,
        grid_widget.param.value,
    )
)

## On-demand consistency checks

These checks run only when the button is clicked. The separate ERA5 850-hPa vorticity file is downloaded lazily for an external reference comparison; it is not treated as exact parity or ground truth.

In [ ]:
def relative_rms(reference: xr.DataArray, candidate: xr.DataArray) -> float:
    a = values(reference)
    b = values(candidate)
    denominator = float(np.sqrt(np.nanmean(a * a)))
    numerator = float(np.sqrt(np.nanmean((a - b) ** 2)))
    if denominator == 0.0:
        return 0.0 if numerator == 0.0 else float("inf")
    return numerator / denominator


def vector_relative_rms(
    reference_u: xr.DataArray,
    reference_v: xr.DataArray,
    candidate_u: xr.DataArray,
    candidate_v: xr.DataArray,
) -> float:
    du = values(reference_u) - values(candidate_u)
    dv = values(reference_v) - values(candidate_v)
    numerator = float(np.sqrt(np.nanmean(du * du + dv * dv)))
    denominator = float(
        np.sqrt(
            np.nanmean(
                values(reference_u) ** 2 + values(reference_v) ** 2
            )
        )
    )
    if denominator == 0.0:
        return 0.0 if numerator == 0.0 else float("inf")
    return numerator / denominator


vo_reference: xr.DataArray | None = None


def load_vo_reference() -> xr.DataArray:
    global vo_reference
    if vo_reference is not None:
        return vo_reference

    with xr.open_dataset(fetch(VO_FILE), engine="h5netcdf") as raw:
        if "vo" not in raw:
            raise ValueError("Pinned ERA5 vorticity file does not contain vo")
        if raw.sizes.get("pressure_level") != 1:
            raise ValueError("Expected one pressure level in ERA5 vo")
        if not np.isclose(pressure_hpa(raw["pressure_level"]), 850.0):
            raise ValueError("Expected ERA5 vo at 850 hPa")
        reference = raw["vo"].isel(pressure_level=0, drop=True).load()

    if not np.array_equal(reference["valid_time"], ERA5["valid_time"]):
        raise ValueError("ERA5 vo timestamps do not match u/v")
    for coordinate in ("latitude", "longitude"):
        if not np.array_equal(reference[coordinate], ERA5[coordinate]):
            raise ValueError(f"ERA5 vo {coordinate} does not match u/v")

    vo_reference = reference
    return vo_reference


def run_checks(_):
    lmin, lmax = map(int, spectral_widget.value_throttled)
    taper = (
        float(taper_value.value_throttled)
        if taper_enabled.value
        else None
    )
    index = int(time_widget.value)
    wind = processed_wind(
        index,
        lmin,
        lmax,
        taper,
        grid_widget.value,
    )
    u, v = wind["u"], wind["v"]

    kin = sg.kinematics(u, v)
    pot = sg.potentials(u, v)
    restored = sg.wind(
        kin["vo"],
        kin["d"],
        source="vorticity_divergence",
    )
    helmholtz = sg.helmholtz(u, v)
    gradient_vp = sg.gradient(pot["vp"])
    inverse_gradient_vp = sg.inverse_gradient(
        gradient_vp["gradient_eastward"],
        gradient_vp["gradient_northward"],
    )
    rotational = sg.rotational_wind(
        kin["vo"],
        quantity="vorticity",
    )
    divergent = sg.divergent_wind(
        kin["d"],
        quantity="divergence",
    )
    vector_lap = sg.vector_laplacian(u, v)
    inverse_vector_lap = sg.inverse_vector_laplacian(
        vector_lap["u"],
        vector_lap["v"],
    )

    metrics = [
        (
            "wind(vo, d) reconstruction",
            vector_relative_rms(
                u,
                v,
                restored["u"],
                restored["v"],
            ),
        ),
        (
            "Helmholtz component sum",
            vector_relative_rms(
                u,
                v,
                helmholtz["u_rotational"] + helmholtz["u_divergent"],
                helmholtz["v_rotational"] + helmholtz["v_divergent"],
            ),
        ),
        (
            "laplacian(strf) versus vo",
            relative_rms(kin["vo"], sg.laplacian(pot["strf"])),
        ),
        (
            "laplacian(vp) versus d",
            relative_rms(kin["d"], sg.laplacian(pot["vp"])),
        ),
        (
            "inverse_laplacian(vo) versus strf",
            relative_rms(
                pot["strf"],
                sg.inverse_laplacian(kin["vo"]),
            ),
        ),
        (
            "inverse_gradient(gradient(vp)) versus vp",
            relative_rms(pot["vp"], inverse_gradient_vp),
        ),
        (
            "rotational + divergent wind",
            vector_relative_rms(
                u,
                v,
                rotational["u_rotational"] + divergent["u_divergent"],
                rotational["v_rotational"] + divergent["v_divergent"],
            ),
        ),
        (
            "vector Laplacian round trip",
            vector_relative_rms(
                u,
                v,
                inverse_vector_lap["u"],
                inverse_vector_lap["v"],
            ),
        ),
    ]

    lines = [
        "### Consistency checks",
        f"Current frame: **{timestamp(index)}**",
        "",
        "Relative RMS is `RMS(a - b) / RMS(a)`.",
        "",
    ]
    lines.extend(
        f"- {name}: `{metric:.3e}`"
        for name, metric in metrics
    )

    try:
        reference = load_vo_reference().isel(valid_time=index)
        calculated = sg.vorticity(*frame(index))
        a = values(reference).ravel()
        b = values(calculated).ravel()
        valid = np.isfinite(a) & np.isfinite(b)
        correlation = float(np.corrcoef(a[valid], b[valid])[0, 1])
        rms_difference = float(
            np.sqrt(np.nanmean((b[valid] - a[valid]) ** 2))
        )
        mean_difference = float(np.nanmean(b[valid] - a[valid]))
        lines.extend(
            [
                "",
                "### External ERA5 `vo` reference",
                f"- correlation: `{correlation:.6f}`",
                f"- RMS difference: `{rms_difference:.3e} s⁻¹`",
                f"- mean difference: `{mean_difference:.3e} s⁻¹`",
            ]
        )
    except (OSError, RuntimeError, ValueError) as exc:
        lines.extend(
            [
                "",
                f"**External ERA5 `vo` comparison unavailable:** {exc}",
            ]
        )

    check_output.object = "\n".join(lines)


check_button = pn.widgets.Button(
    name="Run checks for current frame",
    button_type="primary",
)
check_output = pn.pane.Markdown(
    "Checks run on demand; the separate ERA5 `vo` file is downloaded only here."
)
check_button.on_click(run_checks)

In [ ]:
controls = pn.Column(
    "### Controls",
    diagnostic_widget,
    time_widget,
    timestamp_pane,
    spectral_widget,
    taper_enabled,
    taper_value,
    grid_widget,
    pn.pane.Markdown(
        "Time and numeric spectral controls update after release. "
        "The initial state is frame 0 with a hard `T0–42` selection."
    ),
    width=300,
)

plots = pn.Row(
    pn.Column("### Original", left_plot),
    pn.Column("### Processed", right_plot),
)

main = pn.Column(
    "# spharmgrid — ERA5 850-hPa spherical harmonic explorer",
    "Compare original and processed ERA5 wind on one selected timestamp.",
    plots,
    summary_pane,
    pn.Card(
        check_button,
        check_output,
        title="Consistency checks",
        collapsed=False,
    ),
    pn.pane.Markdown(
        "**Data:** ERA5, Copernicus Climate Change Service / ECMWF; "
        "distributed through PyStormTracker-Data `v0.2.0-data`."
    ),
)

app = pn.Row(
    controls,
    main,
    sizing_mode="stretch_width",
)
app.servable(
    title="spharmgrid — ERA5 850-hPa spherical harmonic explorer"
)
app